# 第67课 · Edit Distance 从零实现 — Levenshtein 动态规划（dynamic programming，DP），WER 的数学基础

**学习目标**
1. 理解编辑距离的三种操作：插入、删除、替换
2. 从零实现 O(m×n) 动态规划算法
3. 用它实现 WER（词错误率）核心计算
4. 明白 ASR 评估为什么离不开它


← **上一课**　[L66 · ASR 系统全览](L66_asr_overview.ipynb)

> 上节课学习了 **ASR 系统全览**：声学模型 → 语言模型 → 解码器，端到端 vs 经典流水线。  
> 本课将探讨 **Edit Distance 从零实现**。

## 本课剧情：Google 翻译怎么知道你打了多少个错别字？

你在搜索框里打了"teh cat sat"，Google 立刻建议"the cat sat"。背后用的是什么算法？

**编辑距离（Edit Distance）**：把一个字符串改成另一个，最少需要几次操作？操作只有三种：

| 操作 | 例子 | 成本 |
|---|---|---|
| 替换（Substitution S） | `teh` → `the`（e↔h互换位置算2次，此处=1次替换h→e）| 1 |
| 插入（Insertion I） | `cat` → `cats`（加s）| 1 |
| 删除（Deletion D） | `cats` → `cat`（去s）| 1 |

**关键洞察**：Levenshtein 动态规划把"两个字符串的对齐问题"分解成**子问题**：

```
dp[i][j] = 将 a[:i] 变成 b[:j] 的最小操作数

if a[i-1] == b[j-1]:  dp[i][j] = dp[i-1][j-1]        # 字符相同，不需操作
else:                  dp[i][j] = 1 + min(
                           dp[i-1][j],    # 删除 a[i-1]
                           dp[i][j-1],    # 插入 b[j-1]
                           dp[i-1][j-1]   # 替换
                       )
```

**Aurora 连接**：`aurora.speech` 的 WER 评估工具正是这个算法的词级版本，用于量化 ASR 系统的输出质量。

本节任务：先实现字符级 `edit_distance(a, b)`，再组合成词级 `wer(reference, hypothesis)`。

In [ ]:
import numpy as np

## ✏️ 任务 1：实现字符级别编辑距离

**实现规范**：

```python
def edit_distance(a, b) -> int:
    # 初始化 (len(a)+1) × (len(b)+1) 的 dp 表
    # dp[i][0] = i（删除 a 的前 i 个字符）
    # dp[0][j] = j（插入 b 的前 j 个字符）
    # 按行填表，返回 dp[m][n]
```

**验收标准**：

| 输入 | 期望输出 | 原因 |
|---|---|---|
| `('', '')` | `0` | 空→空，0步 |
| `('abc', '')` | `3` | 3次删除 |
| `('abc', 'abc')` | `0` | 完全相同 |
| `('kitten', 'sitting')` | `3` | 经典案例 |
| `('', 'abc')` | `3` | 3次插入 |

**注意**：函数接受 `str` 或 `list`（后续 `wer()` 会传词列表）。

In [ ]:
def edit_distance(a, b) -> int:
    """字符级别 Levenshtein 编辑距离，纯 Python 动态规划。

    同样接受词列表（任何可索引序列），因此词级别 WER 也可以复用此函数。

    提示：
      dp[i][j] = 将 a[:i] 变成 b[:j] 的最小编辑次数
      边界：dp[i][0] = i，dp[0][j] = j
      递推：a[i-1]==b[j-1] → dp[i-1][j-1]
             否则 → 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    """
    # TODO: 按上方递推公式填写 DP 表，返回 dp[m][n]
    raise NotImplementedError("你的实现：参考上方 DP 递推公式")


In [ ]:
# ── 参考答案（SOLUTION）─── 完成任务 1 后再展开查看 ──────────────────

def edit_distance(a, b) -> int:  # noqa: F811
    """字符级别 Levenshtein 编辑距离，纯 Python 动态规划。
    同样接受词列表（任何可索引序列）。
    """
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]


In [ ]:
# 验证
try:
    assert edit_distance('', '') == 0
    assert edit_distance('abc', '') == 3
    assert edit_distance('', 'abc') == 3
    assert edit_distance('kitten', 'sitting') == 3   # 经典例子
    assert edit_distance('hello', 'hello') == 0
    print('✅ 字符级别编辑距离验证通过')
except NotImplementedError:
    print('⚠️  edit_distance 尚未实现，请完成任务 1')


## ✏️ 任务 2：实现词级别 WER

**签名**：
```python
def wer(reference: str, hypothesis: str) -> float:
    # reference/hypothesis 是空格分隔的字符串
    # 先 .split()，再调 edit_distance(词列表, 词列表)
    # 返回 edit_distance / len(ref_words)
    # 若 reference 为空返回 0.0
```

**关键设计**：直接复用 `edit_distance()`，只是把"字符"改成"词"。

**验收标准**：

| reference | hypothesis | WER |
|---|---|---|
| `'hello world'` | `'hello world'` | `0.0` |
| `'hello world'` | `'hello'` | `0.5` |
| `'a b c'` | `'a x c'` | `0.333...` |
| `''` | `'anything'` | `0.0` |

**注意**：`len(reference.split())` 就是 N（参考词数）。

In [ ]:
def wer(reference: str, hypothesis: str) -> float:
    """Word Error Rate = 词级别编辑距离 / 参考词数。

    提示：
      1. 将 reference / hypothesis 小写并 .split() 得到词列表
      2. 处理 ref_words 为空的边界情况（见已提供的代码）
      3. 调用 edit_distance(ref_words, hyp_words) 复用 DP——
         edit_distance 对任意可索引序列都适用，无需重写 DP 循环！
      4. 返回 距离 / len(ref_words)
    """
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # TODO: 调用 edit_distance(ref_words, hyp_words) 计算词级别距离
    raise NotImplementedError("你的实现：调用 edit_distance(ref_words, hyp_words)")


In [ ]:
# ── 参考答案（SOLUTION）─── 完成任务 2 后再展开查看 ──────────────────
# 关键设计：edit_distance 接受任意可索引序列，词列表同样适用；
# 直接复用，无需重写 DP 循环——这是正确的代码抽象。

def wer(reference: str, hypothesis: str) -> float:  # noqa: F811
    """Word Error Rate = 词级别编辑距离 / 参考词数。"""
    ref_words = reference.lower().split()
    hyp_words = hypothesis.lower().split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else float('inf')
    # 复用 edit_distance：字符级和词级使用同一 DP 骨架，只换输入序列类型
    return edit_distance(ref_words, hyp_words) / len(ref_words)


In [ ]:
# 验证
try:
    assert wer('hello world', 'hello world') == 0.0
    assert wer('hello world', 'hello') == 0.5        # 1 deletion / 2 words
    assert abs(wer('a b c', 'a b c d') - 1/3) < 1e-9 # 1 insertion / 3 ref words

    ref = 'the cat sat on the mat'
    hyp = 'the cat set on mat'
    print(f'ref: {ref}')
    print(f'hyp: {hyp}')
    print(f'WER = {wer(ref, hyp):.2%}')  # sat->set (S), deleted 'the' (D) = 2/6
    print('✅ WER 验证通过')
except NotImplementedError:
    print('⚠️  wer 尚未实现，请完成任务 2')


## 对齐回溯：可视化每个错误

回溯 dp 表可以知道每个错误是 S（替换）/I（插入）/D（删除）。

In [ ]:
def alignment(ref_str, hyp_str):
    """打印词级别对齐路径，标注 M/S/I/D。"""
    a, b = ref_str.split(), hyp_str.split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    ops, i, j = [], m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and a[i-1] == b[j-1]:
            ops.append(('M', a[i-1], b[j-1])); i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == 1 + dp[i-1][j-1]:
            ops.append(('S', a[i-1], b[j-1])); i -= 1; j -= 1
        elif j > 0 and dp[i][j] == 1 + dp[i][j-1]:
            ops.append(('I', '-', b[j-1])); j -= 1
        else:
            ops.append(('D', a[i-1], '-')); i -= 1
    print(f'{"Op":2s}  {"REF":12s}  {"HYP":12s}')
    print('-' * 30)
    for op, r, h in reversed(ops):
        marker = '  ' if op == 'M' else '**'
        print(f'{marker}{op:2s}  {r:12s}  {h:12s}{marker}')

alignment('the cat sat on the mat', 'the cat set on mat')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| 编辑距离 | 3 种操作，DP 表 (m+1)×(n+1) |
| WER | = 词级编辑距离 / 参考词数，可以 > 1.0 |
| 回溯 | 从 dp[m][n] 往左上角走，还原对齐路径 |
| L68 | CTC 对齐——同样是"所有对齐路径"思想，但面对的是连续帧 |

下一课（L68）将用 CTC 对齐代替手动标注：`ctc_alignment` 把声学帧序列与目标词序列自动对齐，服务于 CTC 系列模型（如 wav2vec 2.0、DeepSpeech）的训练目标。

L69 的 CTC 前向算法也是同一种“填表递推”思想，只是把最小化编辑代价换成 log 概率求和。

## Aurora 连接

本课实现的两个函数对应 `src/aurora/speech/` 生产模块中的 ASR 评估工具：

| 本课函数 | 生产路径 | 说明 |
|---|---|---|
| `edit_distance(a, b)` | `src/aurora/speech/metrics.py::edit_distance` | 泛化版，接受任意序列 |
| `wer(reference, hypothesis)` | `src/aurora/speech/metrics.py::wer` | 委托给 `edit_distance`，无重复 DP |

生产实现的额外特性：
- **O(n) 空间优化**：只需保留相邻两行 `dp[i-1]` 和 `dp[i]`，
  当只需要距离值（不需要回溯路径）时，内存从 O(m×n) 降为 O(n)。
- **批量 WER**：接受多句 `List[str]`，对语料集整体汇报错误率。

```python
# 查看生产模块
from aurora.speech.metrics import edit_distance, wer
print(wer('the cat sat on the mat', 'the cat set on mat'))  # 0.333...
```


## ✏️ 闭卷推导检查格 — Edit Distance DP 表格

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**："cat" → "cut" 的 Levenshtein 距离

1. 写出递推公式（三种操作：插入、删除、替换）
2. 手填完整的 DP 表格（4×4，含边界行/列）
3. 从表格回溯，写出一条最短编辑路径

|   |   | c | u | t |
|---|---|---|---|---|
|   | 0 | 1 | 2 | 3 |
| c | 1 |   |   |   |
| a | 2 |   |   |   |
| t | 3 |   |   |   |

（在此处填表并写路径...）

In [ ]:
# 验证：你填的表格与算法结果一致
import sys; sys.path.insert(0, 'src')

def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n], dp

dist, table = levenshtein("cat", "cut")
assert dist == 1, f"期望距离 1，得到 {dist}"
print("DP 表格（行=cat，列=cut）：")
header = "    " + "  ".join([" "] + list("cut"))
print(header)
for i, row_label in enumerate(" cat"):
    print(f"  {row_label} " + "  ".join(str(table[i][j]) for j in range(4)))
print(f"\n✅ Edit distance('cat','cut') = {dist}")

In [ ]:
# ✏️ 本课自评
l67_review = {
    "levenshtein_dp_memorized":  None,  # 能默写DP转移方程（相同→不变，不同→1+min(删/插/替)）？True/False
    "edit_distance_impl":        None,  # edit_distance()实现正确，5个断言全通过？True/False
    "wer_impl":                  None,  # wer()实现正确，词级版本复用edit_distance？True/False
    "dp_table_traced":           None,  # 白板手推"cat"→"cut"的完整3×3 DP表？True/False
    "whiteboard_passed":         None,  # 白板挑战"cat"→"cut"推导格通关？True/False
}

unfilled = [k for k, v in l67_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l67_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L67 全部通关！进入 L68：CTC 对齐原理')

---

→ **下一课**　[L68 · CTC 对齐原理](L68_ctc_alignment.ipynb)

> 下节课将学习 **CTC 对齐原理**：blank 符号、单调路径与标签折叠。